## Contexte global

Depuis les années 90, il y a eu une véritable prise de conscience mondiale de la nécessité de réduire la consommation d'énergie et des émissions de gaz à effet de serre. Les premiers engagements sont apparus lors de la signature du protocole de Kyoto en 1997. Mais son entrée en vigueur n'a finalement eu lieu qu'en 2005 et de nombreux scientifiques ont jugé les efforts insuffisants pour ralentir le réchauffement climatique. Depuis, d'autres engagements plus ambitieux ont vu le jour (division par 4 des émissions d'ici 2050 pour la France par exemple, engagements de certaines grandes villes comme Paris). Mais la tâche est compliquée. Les pouvoirs publics et les collectivités territoriales n'ont pas la possibilité d'obliger les entreprises et les particuliers à changer leurs habitudes pour atteindre ces objectifs. L'action se porte donc avant tout à faire évoluer les comportements. L'économie et le recyclage des matières premières, l'amélioration des modes de transport et des performances énergétiques des bâtiments doivent devenir des priorités.

## Contexte du projet

l'ADEME a lancé un appel à manifestation d'intérêt pour des démonstrateurs de solutions de mobilité (personnes et marchandises) adaptées à différents territoires. CesiCDP, déjà bien implantée dans la mobilité multimodale intelligente, veut y répondre pour décrocher de nouveaux marchés et financements.

## Problèmatique

Le sujet nous demande de trouver un moyen d'organiser les tournées pour des flottes de camions qui vont dans des villes.

Comment organiser la tournée de livraison des camions de maniere optimale ?

## Livrable final attendu

- ***Version de base***
  - Le choix du modèle et le code en Python capable de résoudre des instances de taille importante (plusieurs milliers de villes)
  - Une étude statistique du comportement expérimental de l'algorithme
  - Une flotte de k camions de capacité identique

- **Version avancée**
  Fenêtre de temps de livraison pour chaque objet
  - Interdiction de livrer hors de la fenêtre
  - Possibilité d'attendre sur place l'ouverture de la fenêtre temporelle

rôle : nous sommes dans l'équipe de 4 personnes chargée de la réponse.

Enjeu métier : les nouvelles technologies de transport, plus économiques et moins polluantes, posent des défis d'optimisation des ressources. La logistique du transport (courrier, livraison, réseau routier, ramassage des ordures) a un impact environnemental important.

Problème à traiter : la gestion de tournées de livraison. Il faut calculer sur un réseau routier une tournée qui relie un sous-ensemble de clients, puis revient au point de départ, en minimisant la durée totale. Le calcul doit tenir compte du trafic prévu sur chaque axe selon la tranche horaire. Cela ressemble à un TSP à durées dépendantes du temps.

Méthode attendue : une approche de Recherche Opérationnelle pour générer la tournée.

Périmètre : il reste à préciser. Vous avez une version de base du problème et hésitez à ajouter des contraintes (fenêtres horaires, capacité, etc.) pour le rendre plus réaliste et plus attractif pour l'ADEME. Ces ajouts le rendraient plus difficile à résoudre, donc il faut arbitrer entre réalisme et faisabilité.

## Données du probleme
- Liste de tous les sommets ($V_0$)
- Liste de tous les clients a livrer + le depot (V = D ∪ N)
- Clients a livrer (N = {1, …, n})
- le nombre de camions (K = {1, …, k})
- capacité du camion (Cmax)
- le poid de chaque arrete, temps entre chaque ville (P(i, j, h))
- Fenettre temporelle ($[a_i, b_i]$)
- Depot (D)
- Demande de chaque client (q_i $\subset q_0 = 0$)

## Variables de décision

x_ijk ∈ {0,1} vaut 1 si le camion k va directement de i à j.

t_i ≥ 0 est l'heure de début de livraison chez le client i (après l'attente éventuelle ; $t_0$ est l'heure de départ du dépôt).

u_i ≥ 0 est la charge cumulée livrée à l'arrivée en i (sert à éviter les sous-tours).


## Contraintes
1. Chaque client est visité exactement une fois

$$\sum_{k \in K}\sum_{j \in V,, j\neq i} x_{ijk} = 1 \qquad \forall i \in N$$

2. Conservation du flux : un camion qui arrive chez un client en repart.

$$\sum_{i \in V, i \neq j} x_{ijk} = \sum_{l \in V, l \neq j} x_{jlk} \qquad \forall j \in N,\ \forall k \in K$$

3. Chaque camion part du dépôt au plus une fois et y revient

$$\sum_{j \in N} x_{0jk} \le 1 \qquad \forall k \in K$$

$$\sum_{i \in N} x_{i0k} = \sum_{j \in N} x_{0jk} \qquad \forall k \in K$$

4. Capacité : la charge totale d'un camion ne dépasse pas Cmax.

$$\sum_{i \in N} q_i \sum_{j \in V} x_{ijk} \le C_{max} \qquad \forall k \in K$$

5. Pas de sous-tours (formulation MTZ, avec u_i la charge cumulée) :

$$u_i - u_j + C_{max}, x_{ijk} \le C_{max} - q_j \qquad \forall i \ne j \in N,\ \forall k \in K$$

$$q_i \le u_i \le C_{max} \qquad \forall i \in N$$

### fenêtres de temps

6. Le temps avance : si le camion k va de i à j, il arrive chez j après être parti de i et avoir roulé.

$$t_j \ge t_i + P_{ij} - M\,(1 - x_{ijk}) \qquad \forall i \in V,\ \forall j \in N,\ i \ne j,\ \forall k \in K$$

Si le camion ne va pas de i à j, $M$ rend la règle toujours vraie.

7. On livre dans la fenêtre du client : pas avant l'ouverture (on attend), pas après la fermeture (interdit).

$$a_i \le t_i \le b_i \qquad \forall i \in N$$

8. Le camion sort du dépôt et rentre dans les horaires du dépôt.

$$a_0 \le t_0 \qquad \qquad t_i + P_{i0} - M\,(1 - x_{i0k}) \le b_0 \qquad \forall i \in N,\ \forall k \in K$$

comme le temps $t$ augmente à chaque trajet ($P_{ij} > 0$), une boucle sans dépôt est impossible. La règle 6 remplace donc la règle 5, qu'on peut enlever.

## Objectif

Optimiser le temps de route totale des camions pendant la tournée

## Complexité

On veut montrer que notre problème est NP-difficile. On le fait par réduction depuis le TSP.

**Problème de décision :** existe-t-il au plus $k$ tournées (départ et retour au dépôt), qui livrent chaque client une fois, avec une charge $\le C_{max}$ par camion et une durée totale $\le B$ ?

**Il est dans NP :** si on nous donne les tournées, on vérifie en temps polynomial que chaque client est livré une fois, que la charge est respectée et que la somme des $P_{ij}$ est $\le B$ (avec les fenêtres, on calcule en plus les heures d'arrivée et on vérifie $a_i \le t_i \le b_i$).

**Il est NP-difficile (réduction depuis le TSP) :**
- Le TSP : on a $n$ villes et des distances $d_{ij}$, existe-t-il un cycle qui passe une fois par chaque ville avec une longueur $\le B$ ? Il est NP-complet.
- À partir d'une instance du TSP, on choisit une ville comme dépôt, les autres sont les clients.
- On prend $k = 1$ camion, $q_i = 1$ et $C_{max} = n - 1$ (la capacité ne gêne jamais), $P_{ij} = d_{ij}$ et la même borne $B$.
- Une tournée du camion unique est exactement un cycle qui passe une fois par chaque ville, de même longueur.

Donc si on savait résoudre notre problème en temps polynomial, on saurait résoudre le TSP. Notre problème est donc NP-difficile, et NP-complet avec la preuve d'appartenance à NP.

La version avancée (avec fenêtres de temps) est au moins aussi dure : en prenant des fenêtres très larges, on retrouve la version de base.

**Remarque :** la capacité apporte une autre difficulté, le bin packing (ranger les clients dans les camions sans dépasser $C_{max}$), qui est aussi NP-difficile.

**Conséquence :** il n'existe pas d'algorithme exact polynomial connu. Le PLNE sert pour les petites instances (solution optimale, borne pour le gap), mais pas pour plusieurs milliers de clients. Pour les grandes instances on utilise une métaheuristique.

### Références

- Karp, R. M. (1972). *Reducibility among combinatorial problems*.
- Garey, M. R. et Johnson, D. S. (1979). *Computers and Intractability: A Guide to the Theory of NP-Completeness*. W. H. Freeman.
- Lenstra, J. K. et Rinnooy Kan, A. H. G. (1981). *Complexity of vehicle routing and scheduling problems*. Networks, 11(2).

## Le PLNE avec PuLP

In [53]:
import pulp

q = {0: 0, 1: 3, 2: 4, 3: 2, 4: 3, 5: 2, 6: 3} # demande de chaque client (q_0 = 0)
Cmax = 10                                      # capacité d'un camion

# matrice d'adjacence
temps = [
    [ 0, 13, 12, 12, 12, 12,  9],
    [13,  0, 17, 25, 24, 16,  9],
    [12, 17,  0, 14, 22, 24,  8],
    [12, 25, 14,  0, 13, 21, 18],
    [12, 24, 22, 13,  0, 13, 21],
    [12, 16, 24, 21, 13,  0, 19],
    [ 9,  9,  8, 18, 21, 19,  0],
]

V = list(range(len(temps))) # tous les sommets (dépôt + clients)
N = V[1:]                   # clients
K = [i for i in range(10)]  # camions
print(V, N, K)

# liste d'adjacence
P = {(i, j): temps[i][j] for i in V for j in V if i != j}
print(P)

[0, 1, 2, 3, 4, 5, 6] [1, 2, 3, 4, 5, 6] [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
{(0, 1): 13, (0, 2): 12, (0, 3): 12, (0, 4): 12, (0, 5): 12, (0, 6): 9, (1, 0): 13, (1, 2): 17, (1, 3): 25, (1, 4): 24, (1, 5): 16, (1, 6): 9, (2, 0): 12, (2, 1): 17, (2, 3): 14, (2, 4): 22, (2, 5): 24, (2, 6): 8, (3, 0): 12, (3, 1): 25, (3, 2): 14, (3, 4): 13, (3, 5): 21, (3, 6): 18, (4, 0): 12, (4, 1): 24, (4, 2): 22, (4, 3): 13, (4, 5): 13, (4, 6): 21, (5, 0): 12, (5, 1): 16, (5, 2): 24, (5, 3): 21, (5, 4): 13, (5, 6): 19, (6, 0): 9, (6, 1): 9, (6, 2): 8, (6, 3): 18, (6, 4): 21, (6, 5): 19}


In [54]:
model = pulp.LpProblem("CVRP", pulp.LpMinimize)

# Variables de décision
arcs = [(i, j, k) for i in V for j in V if i != j for k in K]
x = {(i, j, k): model.add_variable(f"x_{i}_{j}_{k}", cat="Binary") for (i, j, k) in arcs}   # x[i,j,k] = 1 si le camion k va de i à j
u = {i: model.add_variable(f"u_{i}", lowBound=0) for i in N}                                # u[i] = charge cumulée livrée en i

# Objectif
model += pulp.lpSum(P[i, j] * x[i, j, k] for (i, j, k) in arcs)

# Contraintes
# Chaque client est visité exactement une fois
for i in N:
    model += pulp.lpSum(x[i, j, k] for j in V if j != i for k in K) == 1

# Conservation du flux : un camion qui arrive chez un client en repart
for j in N:
    for k in K:
        model += (pulp.lpSum(x[i, j, k] for i in V if i != j)
                  == pulp.lpSum(x[j, l, k] for l in V if l != j))

# Chaque camion part du dépôt au plus une fois et y revient
for k in K:
    model += pulp.lpSum(x[0, j, k] for j in N) <= 1
    model += pulp.lpSum(x[i, 0, k] for i in N) == pulp.lpSum(x[0, j, k] for j in N)

# Capacité : la charge d'un camion ne dépasse pas Cmax
for k in K:
    model += pulp.lpSum(q[i] * x[i, j, k] for i in N for j in V if j != i) <= Cmax

# Pas de sous-tours
for i in N:
    model += u[i] >= q[i]
    model += u[i] <= Cmax
    for j in N:
        if i != j:
            for k in K:
                model += u[i] - u[j] + Cmax * x[i, j, k] <= Cmax - q[j]

In [55]:
resultat = model.solve(pulp.HiGHS(msg=False))
print("Statut :", resultat.status_str)
print("Temps de route total :", round(pulp.value(model.objective), 2))

for k in K:
    suivant = {}
    for i in V:
        for j in V:
            if i != j and x[i, j, k].value() > 0.5:
                suivant[i] = j
    if 0 not in suivant:
        print(f"Camion {k} : reste au dépôt")
        continue
    tournee, i = [0], suivant[0]
    while i != 0:
        tournee.append(i)
        i = suivant[i]
    tournee.append(0)
    charge = sum(q[i] for i in tournee)
    print(f"Camion {k} : {' -> '.join(map(str, tournee))}  (charge {charge}/{Cmax})")

Statut : Optimal
Temps de route total : 92.0
Camion 0 : reste au dépôt
Camion 1 : reste au dépôt
Camion 2 : reste au dépôt
Camion 3 : reste au dépôt
Camion 4 : reste au dépôt
Camion 5 : 0 -> 3 -> 4 -> 5 -> 0  (charge 7/10)
Camion 6 : 0 -> 1 -> 6 -> 2 -> 0  (charge 10/10)
Camion 7 : reste au dépôt
Camion 8 : reste au dépôt
Camion 9 : reste au dépôt
